
# Exercises XP - Diabetes Classification

## What you will learn
- Understanding the problem
- Data collection
- Model training for classification
- Model evaluation

## What you will create
- A Logistic Regression model to predict diabetes



## Exercise 1 - Understanding the problem and Data Collection

We want to predict if an individual has diabetes.

- Load the diabetes dataset and explore it
- Count positive and negative cases
- Split the data into train and test


In [ ]:
import pandas as pd

# Load the CSV file extracted from the zip archive
# Make sure the file is in the same folder as this notebook
df = pd.read_csv('diabetes_prediction_dataset.csv')

# Shape tells us how many rows (patients) and columns (features) we have
print(df.shape)

# head() shows the first 5 rows so we can see what the data looks like
display(df.head())

# dtypes shows the data type of each column (int, float, object/string)
print(df.dtypes)

# isna().sum() counts missing values per column — important before modeling
print("Missing per column:")
display(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Check that the target column 'diabetes' exists in our dataframe
# If this assertion fails, the column name is different — check df.columns
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"

# Count how many patients have diabetes (1) vs do not (0)
# This is important: if one class is much rarer, we have a class imbalance problem
print(df['diabetes'].value_counts())

In [ ]:
# Split the data into features (X) and the target we want to predict (y)
from sklearn.model_selection import train_test_split

# X contains all columns except the target
X = df.drop(columns=['diabetes'])

# y is the column we want to predict: 1 = has diabetes, 0 = no diabetes
y = df['diabetes']

# Split into 80% training data and 20% test data
# stratify=y ensures both splits have the same proportion of 0s and 1s
# random_state=42 makes the split reproducible (same result every time we run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:    ", X_test.shape)


## Exercise 2 - Model picking and standardization

- Which model can we use and why
- Do we need to standardize
- If yes, apply StandardScaler


### Model choice and preprocessing justification

**Logistic Regression** is a great first model for this binary classification task (diabetes: yes or no) because it draws a linear decision boundary between the two classes and outputs calibrated probabilities (values between 0 and 1), which are easy to interpret as a risk score. It is also very interpretable: each feature gets a coefficient that tells us whether it increases or decreases the probability of diabetes, which is valuable in a medical context where we need to explain predictions to doctors or patients.

**Why standardize?** Logistic Regression uses gradient descent internally and is sensitive to the scale of the features. For example, `bmi` ranges roughly from 10 to 95 while `HbA1c_level` ranges from 3.5 to 9. Without scaling, the optimizer would move much faster along the `bmi` dimension, leading to slow convergence or a poorly conditioned solution. Applying `StandardScaler` (which transforms each feature to have mean 0 and standard deviation 1) puts all numeric features on the same scale and ensures the model trains efficiently and converges to a good solution.

**Categorical columns** (`gender`, `smoking_history`) cannot be fed directly to Logistic Regression as text strings — we need to convert them to numbers. We use `OneHotEncoder`, which creates one binary column per category (e.g., `gender_Male`, `gender_Female`), so the model can learn a separate weight for each category without implying any numerical order between them.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Automatically detect which columns are categorical (text) and which are numeric
cat_cols = X.select_dtypes(include=['object']).columns.tolist()      # 'gender', 'smoking_history'
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()  # all numeric columns

# ColumnTransformer applies different transformations to different column groups
# - 'cat': apply OneHotEncoder to text columns (converts categories to 0/1 columns)
#   handle_unknown='ignore' prevents errors if a new category appears at test time
# - 'num': apply StandardScaler to numeric columns (mean=0, std=1)
preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

print("Categorical columns:", cat_cols)
print("Numeric columns:    ", num_cols)

## Exercise 3 - Model training

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# A Pipeline chains steps together: first preprocess, then train the model
# This ensures the scaler is fit only on training data (no data leakage)
clf = Pipeline([
    ('preprocess', preprocess),                                  # step 1: encode + scale
    ('lr', LogisticRegression(max_iter=1000, random_state=42))   # step 2: logistic regression
    # max_iter=1000 gives the optimizer enough iterations to converge
])

# fit() trains the entire pipeline on the training data
clf.fit(X_train, y_train)

print("Model training complete!")


## Exercise 4 - Evaluation metrics

- Plot accuracy and comment
- Plot confusion matrix and comment
- Plot precision, recall, F1 and comment


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Use the trained pipeline to predict labels (0 or 1) for the test set
y_pred = clf.predict(X_test)

# ---- Compute all metrics ----
acc  = accuracy_score(y_test, y_pred)                           # overall % correct
prec = precision_score(y_test, y_pred)                          # of predicted diabetics, how many truly are?
rec  = recall_score(y_test, y_pred)                             # of true diabetics, how many did we catch?
f1   = f1_score(y_test, y_pred)                                 # harmonic mean of precision and recall

print("Accuracy: ", round(acc,  4))
print("Precision:", round(prec, 4))
print("Recall:   ", round(rec,  4))
print("F1:       ", round(f1,   4))

# ---- Bar chart of the four metrics ----
# A quick visual to compare all metrics at a glance
plt.figure(figsize=(6, 4))
bars = plt.bar(['Accuracy', 'Precision', 'Recall', 'F1'],
               [acc, prec, rec, f1],
               color=['steelblue', 'coral', 'mediumpurple', 'mediumseagreen'])
plt.ylim(0, 1.05)
plt.title('Model metrics on test set')
plt.ylabel('Score')
# Add score labels on top of each bar
for bar, val in zip(bars, [acc, prec, rec, f1]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

# ---- Confusion matrix ----
# Rows = actual class, Columns = predicted class
# Top-left: True Negatives (correctly predicted no diabetes)
# Top-right: False Positives (said diabetic but they are not)
# Bottom-left: False Negatives (missed a real diabetic — costly in medicine!)
# Bottom-right: True Positives (correctly predicted diabetes)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

### Comment on precision vs recall

The dataset is **highly imbalanced**: about 91.5% of patients do not have diabetes and only 8.5% do. This means a naive model that always predicts "no diabetes" would achieve ~91.5% accuracy — but would be completely useless clinically because it would miss every real diabetic patient.

That is why **Recall** (also called Sensitivity) is the most important metric here. Recall measures the proportion of actual diabetic patients that our model correctly identifies. Missing a diabetic patient (a False Negative) is a serious medical error — the patient goes untreated. We want Recall to be as high as possible.

**Precision** tells us, among the patients we flagged as diabetic, how many actually are. Low precision means many False Positives (healthy people sent for unnecessary tests), which wastes resources and causes patient anxiety — but is less dangerous than missing a true case.

The **F1-score** balances both Precision and Recall into a single number. If Recall is much lower than Precision, we may want to lower the decision threshold (currently 0.5) so the model flags more patients as diabetic, trading some Precision for better Recall.


## Exercise 5 - Visualizing the performance of our model

Visualize a 2D decision boundary with accuracy info. Use only two informative features for this plot to keep it 2D. Suggested pair: `HbA1c_level` and `blood_glucose_level` if present. Otherwise pick any two numeric features.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Pick two features that are medically meaningful for diabetes diagnosis:
# HbA1c_level (average blood sugar over 3 months) and blood_glucose_level (current blood sugar)
feat_x = 'HbA1c_level'         if 'HbA1c_level'         in X.columns else X.select_dtypes(include=['int64','float64']).columns[0]
feat_y = 'blood_glucose_level'  if 'blood_glucose_level'  in X.columns else X.select_dtypes(include=['int64','float64']).columns[1]

# Keep only those two features — we need 2D for the plot
X2_train = X_train[[feat_x, feat_y]].copy()
X2_test  = X_test[[feat_x, feat_y]].copy()

# Train a simpler pipeline using only these 2 features
pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0, 1])], remainder='drop')),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Build a grid of points covering the feature space
# We will predict the diabetes probability for every point in this grid
x_min, x_max = X2_train[feat_x].min() - 1, X2_train[feat_x].max() + 1
y_min, y_max = X2_train[feat_y].min() - 1, X2_train[feat_y].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

# Predicted probability of diabetes for every grid point
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(7, 5))

# Draw the decision boundary (the contour where predicted probability = 0.5)
# Points above this line are classified as diabetic, points below as non-diabetic
cs = plt.contour(xx, yy, probs, levels=[0.5])
plt.clabel(cs, inline=True, fmt={0.5: 'P=0.5'})

# Scatter the test points, colored by true label (0 = no diabetes, 1 = diabetes)
plt.scatter(X2_test[feat_x], X2_test[feat_y], c=y_test, alpha=0.4, cmap='coolwarm', s=5)

plt.xlabel(feat_x)
plt.ylabel(feat_y)

acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))
plt.title(f'Decision boundary (2 features) — test accuracy {acc2:.3f}')
plt.tight_layout()
plt.show()

# Note: accuracy is lower here because we only used 2 features instead of all 8.
# The full model (clf) uses all features and should perform better.


## Exercise 6 - ROC curve

Use the code template provided to plot the ROC curve for your model and compute AUC. You can reuse the fitted `clf` pipeline.

Template summary:
- Get predicted probabilities for the positive class
- Compute fpr and tpr with `roc_curve`
- Plot ROC and print AUC


In [ ]:
import matplotlib.pyplot as plt
from sklearn import metrics

# predict_proba returns [prob_class_0, prob_class_1] for each row
# We take column index 1 = probability that the patient HAS diabetes
y_proba = clf.predict_proba(X_test)[:, 1]

# roc_curve computes True Positive Rate (Recall) and False Positive Rate
# at many different probability thresholds, not just 0.5
fpr, tpr, _ = metrics.roc_curve(y_test, y_proba)

# AUC (Area Under the Curve): ranges from 0.5 (random) to 1.0 (perfect)
# It measures the model's ability to rank a random diabetic patient
# higher than a random non-diabetic patient
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})', color='steelblue', lw=2)

# The dashed diagonal represents a random classifier (AUC = 0.5)
# Our model should be well above this line
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.title('ROC Curve — Diabetes Classification')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"AUC = {auc:.4f}")

### Interpretation of the ROC curve and AUC

The ROC curve plots the **True Positive Rate** (how many real diabetics we correctly identify) against the **False Positive Rate** (how many healthy patients we incorrectly flag as diabetic) at every possible decision threshold between 0 and 1. The curve that bows strongly toward the top-left corner is a sign of a good model: it catches most diabetics (high TPR) while keeping false alarms low (low FPR).

The **AUC (Area Under the Curve)** summarises this into a single number. An AUC of **0.5** means the model is no better than random guessing (the dashed diagonal). An AUC of **1.0** would be a perfect model. An AUC in the range **0.90–0.97** (which is typical for Logistic Regression on this dataset) means the model is very good at distinguishing diabetic from non-diabetic patients.

Practically, a high AUC means we can choose a threshold that gives us a good balance of sensitivity (catching real diabetics) and specificity (not over-alarming healthy patients). For a medical screening tool, we would typically lower the threshold below 0.5 to prioritise recall — it is safer to investigate a few extra healthy patients than to miss a true diabetic.